# Customer Churn Prediction — EDA

This notebook covers the initial exploratory data analysis (EDA) 
for the customer churn dataset. The goal is to understand the 
structure, quality, and distributions of the data before any 
modeling.

In [7]:
#Libraries

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
#Load data
DATA_PATH = Path("..") / "data" / "raw" / "customer_churn.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print(f"Shape: {df.shape}")

Dataset loaded successfully
Shape: (100000, 32)


## 1. Data Loading

The dataset contains **100,000 customers** and **32 features**, 
including behavioral, financial, and support-related variables. 
The target variable is `churn` (1 = churned, 0 = retained).

In [11]:
#General information

print("First 5 rows\n\n")
print(df.head())

print("Data types\n\n")
print(df.dtypes)

print("Basic information\n\n")
print(df.info())

First 5 rows


   customer_id  gender  age     country      city customer_segment  \
0  CUST_000001    Male   68  Bangladesh    London              SME   
1  CUST_000002  Female   57      Canada    Sydney       Individual   
2  CUST_000003    Male   24     Germany  New York              SME   
3  CUST_000004    Male   49   Australia     Dhaka       Individual   
4  CUST_000005    Male   65  Bangladesh     Delhi       Individual   

   tenure_months signup_channel contract_type  monthly_logins  ...  \
0             22            Web       Monthly              26  ...   
1              9         Mobile       Monthly               7  ...   
2             58            Web        Yearly              19  ...   
3             19         Mobile        Yearly              34  ...   
4             52            Web       Monthly              20  ...   

   avg_resolution_time  complaint_type  csat_score  escalations  \
0            13.354360         Service         4.0            0   
1        

## 2. First Inspection

The dataset has **32 columns** split across three data types:
- **14 integer columns** — logins, tickets, tenure, churn target
- **6 float columns** — session time, rates, scores
- **12 object columns** — categorical variables like gender, country, contract type


In [13]:
#Missing values

missing=df.isnull().sum()
missing_percentage=(missing/len(df)*100).round(2)

missing_df=pd.DataFrame({
    "missing_count":missing,
    "missing _percentage":missing_percentage
})

missing_df=missing_df[missing_df["missing_count"]>0]

print("Columns with missing values")
print(missing_df)

Columns with missing values
                missing_count  missing _percentage
complaint_type          20534                20.53


In [15]:
# Investigating complaint_type nulls 
mask_null = df["complaint_type"].isnull()

print("Clients WITH complaint_type null:")
print(df[mask_null]["support_tickets"].describe())

print("\nClients WITHOUT complaint_type nulL:")
print(df[~mask_null]["support_tickets"].describe())

Clients WITH complaint_type null:
count    20534.000000
mean         1.197234
std          1.094769
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max          7.000000
Name: support_tickets, dtype: float64

Clients WITHOUT complaint_type nulL:
count    79466.000000
mean         1.210568
std          1.104982
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max          7.000000
Name: support_tickets, dtype: float64


## 3. Missing Values

Only one column has missing values: `complaint_type` with **20,534 nulls (20.53%)**.

Investigation shows that customers with null `complaint_type` have a 
similar ticket volume (mean: 1.20) to those without nulls (mean: 1.21). 
This suggests the nulls represent **unclassified complaints**, not 
absence of complaints.

These nulls will be treated as a separate category `"Unknown"` during 
the data cleaning phase.

In [19]:
# Churn Distribution 
churn_counts = df["churn"].value_counts()
churn_percentage = df["churn"].value_counts(normalize=True).mul(100).round(2)

churn_summary = pd.DataFrame({
    "count": churn_counts,
    "percentage": churn_percentage
})

churn_summary.index = ["Retained (0)", "Churned (1)"]

print("Churn Distribution")
print(churn_summary)

Churn Distribution
              count  percentage
Retained (0)  89863       89.86
Churned (1)   10137       10.14


## 4. Churn Distribution

The dataset is heavily imbalanced:
- **Retained (0): 89,863 customers — 89.86%**
- **Churned (1): 10,137 customers — 10.14%**

This is a critical finding. A naive model that always predicts 
"retained" would achieve ~90% accuracy but zero business value.

This imbalance will be addressed during modeling using techniques 
such as SMOTE or `class_weight` adjustments.

In [ ]:
# Descriptive Statistics 
desc = df.describe().T

descriptive_stats = df.describe().T

descriptive_stats = descriptive_stats[["mean", "std", "min", "50%", "max"]]

descriptive_stats.columns = ["mean", "std_dev", "min", "median", "max"] #Colums been renamed

descriptive_stats = descriptive_stats.round(2)

print("Descriptive Statistics")
print(descriptive_stats)

Descriptive Statistics
                         mean  std_dev     min  median      max
age                     45.83    16.40   18.00   45.00    74.00
tenure_months           30.15    17.08    1.00   30.00    59.00
monthly_logins          19.65     9.83    0.00   20.00    54.00
weekly_active_days       3.48     2.30    0.00    3.00     7.00
avg_session_time        15.20     6.84    1.00   15.18    42.00
features_used            4.97     2.21    1.00    5.00    15.00
usage_growth_rate        0.02     0.15   -0.58    0.02     0.54
last_login_days_ago      9.50     9.75    0.00    7.00    80.00
monthly_fee             35.04    23.83   10.00   30.00   100.00
total_revenue         1054.10  1018.85   10.00  720.00  5900.00
payment_failures         0.50     0.71    0.00    0.00     5.00
support_tickets          1.21     1.10    0.00    1.00     7.00
avg_resolution_time     23.99     9.97    1.00   24.02    61.82
csat_score               3.49     0.98    1.00    4.00     5.00
escalations      

## 5. Descriptive Statistics

Key observations:
- `days_since_last_login`: max of 80 days suggests disengaged customers
- `payment_failures`: some customers have up to 5 failed payments
- `usage_growth_rate`: negative values indicate declining engagement
- `total_revenue`: high std_dev (1018) reveals very diverse customer value